In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn

In [2]:
from ast import literal_eval

df = pd.read_csv('data/matches_processed.csv')
df["Player"] = df["Player"].apply(literal_eval)

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder

class WinPredictionDataset(Dataset):
    def __init__(self, players, result):
        self.players = players
        self.results = torch.tensor(result, dtype=torch.float32)
        
        # Flatten all tokens to build vocabulary
        all_tokens = [item for row in players for subsublist in row for item in subsublist]
        self.tokenizer = LabelEncoder()
        self.tokenizer.fit(all_tokens)  # Fit on all possible tokens
        
        # Pre-encode all data during init (more efficient)
        self.encoded_players = [
            [
                self.tokenizer.transform(subsublist) 
                for subsublist in row
            ] 
            for row in players
        ]
        
    def __len__(self):
        return len(self.players)

    def __getitem__(self, idx):
        return {
            "players": torch.tensor(self.encoded_players[idx], dtype=torch.long),  # Shape: [10, 5]
            "results": self.results[idx]  # Shape: [1]
        }

In [30]:
import torch
import torch.nn as nn

class WinPredictionModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, encoder_dim=64):
        super().__init__()
        
        # 1. Embedding Layer 
        self.embedding = nn.Embedding(vocab_size + 1, embed_dim, padding_idx=0)
        
        # 2. 2D CNN (Spatial Feature Extraction)
        self.cnn = nn.Sequential(
            nn.Conv2d(1, 16, kernel_size=(3,3), padding=1),  # [batch, 32, 10, 5]
            nn.BatchNorm2d(16),
            nn.Dropout(0.2),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),  # [batch, 32, 5, 2]
        )
        
        # 3. Encoder (Processes CNN outputs)
        self.encoder = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=16*2,  # Flattened CNN channels
                nhead=8,
                dim_feedforward=encoder_dim,
                dropout=0.2,
            ),
            num_layers=1
        )
        
        # 4. Classifier
        self.classifier = nn.Sequential(
            nn.Linear(16*2*5, 64),  # 5 players after pooling
            nn.Sigmoid(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        # x shape: [batch_size, 10, 5]
        
        # 1. Embed tokens
        x = self.embedding(x)  # [batch, 10, 5, embed_dim]
        
        # 2. Prepare for CNN (average embeddings)
        x = x.mean(dim=-1, keepdim=True)  # [batch, 10, 5, 1]
        x = x.permute(0, 3, 1, 2)  # [batch, 1, 10, 5]
        
        # 3. 2D CNN
        cnn_out = self.cnn(x)  # [batch, 32, 5, 2]
        
        # 4. Prepare for encoder
        batch_size, channels, h, w = cnn_out.shape
        cnn_flat = cnn_out.reshape(batch_size, h, channels*w)  # [batch, 5, 64]
        
        # 5. Transformer encoder
        encoded = self.encoder(cnn_flat)  # [batch, 5, 64]
        
        # 6. Classifier
        out = self.classifier(encoded.reshape(batch_size, -1))
        return torch.sigmoid(out)

In [5]:
import plotly.graph_objects as go

def visualize_losses(train_losses, val_losses):
    fig = go.Figure()
        
    fig.add_trace(
        go.Scatter(
            x=list(range(1, len(train_losses) + 1)),
            y=train_losses,
            mode='lines',
            name='Train Loss',
            line=dict(color='blue'),
        )
    )
    
    fig.add_trace(
        go.Scatter(
            x=list(range(1, len(val_losses) + 1)),
            y=val_losses,
            mode='lines',
            name='Validation Loss',
            line=dict(color='orange'),
        )
    )
    
    fig.update_layout(
        title="Training and Validation Loss",
        xaxis_title="Epochs",
        yaxis_title="Mean Rowwise RMSE",
        template="plotly_white"
    )

    fig.show()

In [ ]:
def eval_model(model, val_loader, criterion, device):
    model.eval()

    # Disable gradient computation during testing (saves memory and computation)
    with torch.no_grad():
        val_loss = 0.0
        for batch in val_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            # Forward pass
            outputs = model(players)
            loss = criterion(torch.round(outputs), results.unsqueeze(1))
            
            val_loss += loss.item()

    return val_loss / len(val_loader)

In [31]:
from torch.utils.data import DataLoader
from sklearn.model_selection import KFold
import torch
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

dataset = WinPredictionDataset(df["Player"].values, df["Win"].to_numpy())

kf = KFold(n_splits=5, shuffle=True, random_state=42)

batch_size = 64

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_epochs = 10

for fold, (train_idx, val_idx) in enumerate(kf.split(dataset)):
    print(f"Fold {fold + 1}/{kf.n_splits}")
    
    trainset = torch.utils.data.Subset(dataset, train_idx)
    valset = torch.utils.data.Subset(dataset, val_idx)

    train_loader = DataLoader(trainset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(valset, batch_size=batch_size, shuffle=False)
    
    # Initialize model, optimizer, and loss function
    model = WinPredictionModel(vocab_size=dataset.tokenizer.classes_.size).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCELoss()

    train_losses = []
    val_losses = []
    for epoch in range(num_epochs):
        train_loss = 0
        model.train()
        for batch in train_loader:
            players = batch["players"].to(device)
            results = batch["results"].to(device)
            
            optimizer.zero_grad()
            outputs = model(players)
            loss = criterion(outputs, results.unsqueeze(1))  # Ensure results are the same shape as outputs
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        average_train_loss = train_loss / len(train_loader)
        average_val_loss = eval_model(model, val_loader, criterion, device)
        
        print(f"Epoch {epoch + 1}/{num_epochs}, Average Train Loss: {average_train_loss}, Average Val Loss: {average_val_loss}")
        
        train_losses.append(average_train_loss)
        val_losses.append(average_val_loss)

    visualize_losses(train_losses, val_losses)

Using device: cuda
Fold 1/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6914007258415222, Average Val Loss: 0.6920362934470177
Epoch 2/10, Average Train Loss: 0.6861115427017211, Average Val Loss: 0.6931434608995914
Epoch 3/10, Average Train Loss: 0.6825106496810913, Average Val Loss: 0.7036602180451155
Epoch 4/10, Average Train Loss: 0.6773569750785827, Average Val Loss: 0.69025425799191
Epoch 5/10, Average Train Loss: 0.6722439451217651, Average Val Loss: 0.6925276219844818
Epoch 6/10, Average Train Loss: 0.6687413125038147, Average Val Loss: 0.6919053755700588
Epoch 7/10, Average Train Loss: 0.6670731387138367, Average Val Loss: 0.6998843085020781
Epoch 8/10, Average Train Loss: 0.6652671875953674, Average Val Loss: 0.6890042573213577
Epoch 9/10, Average Train Loss: 0.6627789211273193, Average Val Loss: 0.6943517494946718
Epoch 10/10, Average Train Loss: 0.6592484111785889, Average Val Loss: 0.693001251667738


Fold 2/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



...

c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6922199158668518, Average Val Loss: 0.6870957501232624
Epoch 2/10, Average Train Loss: 0.6892600102424622, Average Val Loss: 0.6842806655913591
Epoch 3/10, Average Train Loss: 0.682757749080658, Average Val Loss: 0.6780835743993521
Epoch 4/10, Average Train Loss: 0.6768963704109192, Average Val Loss: 0.681316526606679
Epoch 5/10, Average Train Loss: 0.6723055119514465, Average Val Loss: 0.679655933752656
Epoch 6/10, Average Train Loss: 0.6705599012374878, Average Val Loss: 0.6786068305373192
Epoch 7/10, Average Train Loss: 0.6662578692436218, Average Val Loss: 0.676677618175745
Epoch 8/10, Average Train Loss: 0.6654182562828064, Average Val Loss: 0.6745446566492319
Epoch 9/10, Average Train Loss: 0.6617661633491516, Average Val Loss: 0.677989337593317
Epoch 10/10, Average Train Loss: 0.6614712252616882, Average Val Loss: 0.6789378505200148


Fold 5/5


c:\Users\jager\Desktop\github\win_prediction\venv\Lib\site-packages\torch\nn\modules\transformer.py:385: UserWarning:

enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)



Epoch 1/10, Average Train Loss: 0.6924881620407104, Average Val Loss: 0.6901733949780464
Epoch 2/10, Average Train Loss: 0.6903585510253907, Average Val Loss: 0.6924319826066494
Epoch 3/10, Average Train Loss: 0.6862412700653077, Average Val Loss: 0.6901712864637375
Epoch 4/10, Average Train Loss: 0.6840639872550964, Average Val Loss: 0.7047672402113676
Epoch 5/10, Average Train Loss: 0.6826840238571167, Average Val Loss: 0.6931653395295143
Epoch 6/10, Average Train Loss: 0.6792149176597595, Average Val Loss: 0.6958744488656521
Epoch 7/10, Average Train Loss: 0.6739232039451599, Average Val Loss: 0.697100842371583
Epoch 8/10, Average Train Loss: 0.6727870259284973, Average Val Loss: 0.697851886972785
Epoch 9/10, Average Train Loss: 0.6697611570358276, Average Val Loss: 0.6970975957810879
Epoch 10/10, Average Train Loss: 0.666464207649231, Average Val Loss: 0.6988602746278048
